In [1]:
import pandas as pd
import re
import numpy as np
from datetime import datetime, timedelta
from gspread_pandas import Spread, conf

In [20]:
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
# 3. Define the Sheet URL (Name - Dispapatch list KSA)
logistic_sheet_url = "https://docs.google.com/spreadsheets/d/1eo2tY_57lcOTtGOyU5GOz2IQON3b0VsVf8rTsbfT3HM/edit?gid=0#gid=0"
target_sheet_url = "https://docs.google.com/spreadsheets/d/1ndt_6iIRLEihA8OOA8GyaS0GkpuYTSU3ZZfZT75JkkU/edit?gid=1468096196#gid=1468096196"

In [22]:
c = conf.get_config(file_name=config_path)
spread = Spread(logistic_sheet_url, config=c)
target_spread = Spread(target_sheet_url, config=c)


In [4]:
try:
    df = spread.sheet_to_df(
        sheet="Order list", 
        index=None, 
    )
    
    print(f"Successfully loaded {len(df)} rows of agent data.")

except Exception as e:
    print(f"Error: {e}")

Successfully loaded 936 rows of agent data.


In [5]:
def safe_upload(spreadsheet, df, sheet_name):
    if not df.empty:
        spreadsheet.open_sheet(sheet_name, create=True)
        spreadsheet.sheet.batch_clear(['A2:Z10000']) 
        upload_df = df.astype(str)
        spreadsheet.df_to_sheet(upload_df, index=False, replace=False, headers=False, start="A2")

In [6]:
# if df.empty:
 
df.columns = df.columns.astype(str).str.strip().str.upper()

if 'DATE' not in df.columns:
    print(f"Error: 'DATE' column missing.")
        # return
            
df['DATE'] = pd.to_datetime(df['DATE'], dayfirst=True, errors='coerce').dt.date
df = df.dropna(subset=['DATE'])

if 'TRACKING NUMBER' in df.columns:
    df = df[df['TRACKING NUMBER'].astype(str).str.strip() == ""]




In [7]:
len(df)

135

In [8]:
df.columns

Index(['SL NO', 'AGENT', 'DATE', 'TRACKING NUMBER', 'EM NUMBER', 'NAME',
       'NUMBER1', 'NUMBER2', 'STATE/CITY', 'ADDRESS', 'PRODUCT1', 'QTY',
       'PRODUCT2', 'QTY', 'TOTAL', 'PAYMENT METHOD', 'REMARKS',
       'NATIONAL CODE', 'COUNTRY', 'DELIVERY AGENT', 'DISPATCHED DATE',
       'STATUS', 'DELIVERED/CANCELLED DATE', 'DELAYED REASON',
       'CANCELLATION REASON', 'ZONE', 'WILAYAT', 'NOTES'],
      dtype='str')

In [9]:
KSA_df = df[df['COUNTRY'] == 'KSA'].copy()
UAE_df = df[df['COUNTRY'] == 'UAE'].copy()
QATAR_df = df[df['COUNTRY'] == 'QATAR'].copy()
BAHRAIN_df = df[df['COUNTRY'] == 'BAHRAIN'].copy()

In [13]:


# Function to validate UAE mobile numbers
def is_uae_number(phone):
    if pd.isna(phone):
        return False
    # Clean the number (remove non-digits like '+')
    clean_phone = re.sub(r'\D', '', str(phone))
    

    pattern = r'^(?:971|0)?5[024568]\d{7}$'
    return bool(re.match(pattern, clean_phone))

# 1. Filter UAE_df to keep ONLY valid UAE numbers
UAE_df_filtered = UAE_df[UAE_df['NUMBER1'].apply(is_uae_number)].copy()

# 2. Define the exact columns and order from the image
target_columns = [
    'CUSTOMER_NAME', 'MOBILE_NO', 'LANDLINE_NO', 'ADDRESS_1', 
    'ADDRESS_2', 'ADDRESS_3', 'FLAT/VILLANO', 'DELIVERY_CITY', 
    'COD_AMOUNT', 'REMARKS', 'REFERENCE_NO', 'OTHER_REMARKS'
]

# 3. Create the formatted DataFrame
formatted_UAE = pd.DataFrame()

formatted_UAE['CUSTOMER_NAME'] = UAE_df_filtered['NAME']
formatted_UAE['MOBILE_NO']     = UAE_df_filtered['NUMBER1']
formatted_UAE['LANDLINE_NO']   = UAE_df_filtered['NUMBER2']
formatted_UAE['ADDRESS_1']     = UAE_df_filtered['ADDRESS']
formatted_UAE['ADDRESS_2']     = "" # Empty column
formatted_UAE['ADDRESS_3']     = "" # Empty column
formatted_UAE['FLAT/VILLANO']  = "" # Empty column
formatted_UAE['DELIVERY_CITY'] = UAE_df_filtered['STATE/CITY']
formatted_UAE['COD_AMOUNT']    = UAE_df_filtered['TOTAL']
formatted_UAE['REMARKS']       = UAE_df_filtered['PRODUCT1']
formatted_UAE['REFERENCE_NO']  = UAE_df_filtered['EM NUMBER']
formatted_UAE['OTHER_REMARKS'] = UAE_df_filtered['NOTES']

# Ensure the columns are in the exact order shown in the image
formatted_UAE = formatted_UAE[target_columns]

# Show result
formatted_UAE.head(2)

,CUSTOMER_NAME,MOBILE_NO,LANDLINE_NO,ADDRESS_1,ADDRESS_2,ADDRESS_3,FLAT/VILLANO,DELIVERY_CITY,COD_AMOUNT,REMARKS,REFERENCE_NO,OTHER_REMARKS
162,Vijesh Mullakal,971521602642,,M 41 St 8/9Abu DhabiUnited Arab Emirates,,,,ABU DHABI,120,DOE COLLECTION,EMCR21574,
168,Amir Khan,971562236412,,mussafah 32 abudhabi,,,,Abudhabi,130,AMBER,EMCR18562,


In [14]:
def is_qatar_number(phone):
    if pd.isna(phone):
        return False
    # Clean the number
    clean_phone = re.sub(r'\D', '', str(phone))
    
    # Logic: Optional 974 or 0, then starts with 3, 5, 6, or 7, followed by 7 digits (8 total)
    pattern = r'^(?:974|0)?([3567]\d{7})$'
    return bool(re.match(pattern, clean_phone))

def get_clean_qatar_number(phone):
    if pd.isna(phone):
        return None
    
    # Remove all non-digits
    clean_phone = re.sub(r'\D', '', str(phone))
    
    # Pattern: Optional 974 or 0, then capture the 8-digit local number
    # Local numbers in Qatar start with 3, 5, 6, or 7
    pattern = r'^(?:974|0)?([3567]\d{7})$'
    match = re.match(pattern, clean_phone)
    
    if match:
        return match.group(1)  # Returns ONLY the 8-digit part
    return None

# Create a temporary cleaned column to filter and store the 8-digit number
QATAR_df['CLEAN_LOCAL_NO'] = QATAR_df['NUMBER1'].apply(get_clean_qatar_number)

# 1. Filter: Keep only rows where a valid 8-digit number was extracted
QATAR_df_filtered = QATAR_df[QATAR_df['CLEAN_LOCAL_NO'].notna()].copy()

# 2. Define the exact columns (same as your list)
target_columns = [
    "client_order_ref", "customer_name", "partner_id", "whatsapp_no", "source_id", 
    "Pricelist Name", "street_no", "building_no", "zone_id (governarate)", 
    "wilayat_id", "city_id", "order_line/product_id", "order_line/product_uom", 
    "order_line/price_unit", "order_line/product_uom_qty remarks"
]

# 3. Create the formatted DataFrame
formatted_QATAR = pd.DataFrame()

# Mapping based on your provided column list
formatted_QATAR['client_order_ref'] = QATAR_df_filtered['EM NUMBER']
formatted_QATAR['customer_name']    = QATAR_df_filtered['NAME']
formatted_QATAR['partner_id']      = QATAR_df_filtered['NUMBER1']
formatted_QATAR['whatsapp_no']     = QATAR_df_filtered['NUMBER1']
formatted_QATAR['source_id']       = "SCENT PASSION - BAHRAIN" 
formatted_QATAR['Pricelist Name']  = "Default BHD pricelist" 
formatted_QATAR['street_no']       = QATAR_df_filtered['ADDRESS']
formatted_QATAR['building_no']     = "" # From your list: 'FLAT/VILLANO' or empty
formatted_QATAR['zone_id (governarate)'] = QATAR_df_filtered['ZONE']
formatted_QATAR['wilayat_id']      = QATAR_df_filtered['WILAYAT']
formatted_QATAR['city_id']         = QATAR_df_filtered['STATE/CITY']
formatted_QATAR['order_line/product_id']  = QATAR_df_filtered['PRODUCT1']
formatted_QATAR['order_line/product_uom'] = "Unit" # Example static value
formatted_QATAR['order_line/price_unit']  = QATAR_df_filtered['TOTAL']
formatted_QATAR['order_line/product_uom_qty remarks'] = QATAR_df_filtered['REMARKS']

# Ensure the columns are in the exact order requested
formatted_QATAR = formatted_QATAR[target_columns]

# Show result
formatted_QATAR.head(2)

,client_order_ref,customer_name,partner_id,whatsapp_no,source_id,Pricelist Name,street_no,building_no,zone_id (governarate),wilayat_id,city_id,order_line/product_id,order_line/product_uom,order_line/price_unit,order_line/product_uom_qty remarks
693,EMCR7889,usmanmuzammilm,31579897,31579897,SCENT PASSION - BAHRAIN,Default BHD pricelist,"Hilal\nNew salata\nAll aRab stadium6GWM+J9V, D...",,,,Hilal,ESENCIA FLORAL,Unit,130,HE WANT THE DELIVERY ON FEBRUARY 28
735,EMCR11065,Deepak Krishna,70373080,70373080,SCENT PASSION - BAHRAIN,Default BHD pricelist,"https://maps.app.goo.gl/22htJfNCU3DiAd398, 5CX...",,,,Doha,ESENCIA FLORAL,Unit,130,


In [23]:
import pandas as pd
import re

def get_clean_bahrain_number(phone):
    if pd.isna(phone):
        return None
 
    clean_phone = re.sub(r'\D', '', str(phone))

    pattern = r'^(?:973|0)?([136]\d{7})$'
    match = re.match(pattern, clean_phone)
    
    if match:
        return match.group(1) 
    return None


BAHRAIN_df['CLEAN_LOCAL_NO'] = BAHRAIN_df['NUMBER1'].apply(get_clean_bahrain_number)
BAHRAIN_df_filtered = BAHRAIN_df[BAHRAIN_df['CLEAN_LOCAL_NO'].notna()].copy()

formatted_BAHRAIN = pd.DataFrame()

formatted_BAHRAIN['client_order_ref'] = BAHRAIN_df_filtered['EM NUMBER']
formatted_BAHRAIN['customer_name']    = BAHRAIN_df_filtered['NAME']
formatted_BAHRAIN['partner_id']       = BAHRAIN_df_filtered['CLEAN_LOCAL_NO']
formatted_BAHRAIN['whatsapp_no']      = BAHRAIN_df_filtered['CLEAN_LOCAL_NO']
formatted_BAHRAIN['source_id']        = "SCENT PASSION - BAHRAIN" 
formatted_BAHRAIN['Pricelist Name']   = "Default BHD pricelist" 
formatted_BAHRAIN['street_no']        = BAHRAIN_df_filtered['ADDRESS']
formatted_BAHRAIN['building_no']      = "" 
formatted_BAHRAIN['zone_id (governarate)'] = BAHRAIN_df_filtered['ZONE']
formatted_BAHRAIN['wilayat_id']       = BAHRAIN_df_filtered['WILAYAT']
formatted_BAHRAIN['city_id']          = BAHRAIN_df_filtered['STATE/CITY']
formatted_BAHRAIN['order_line/product_id']  = BAHRAIN_df_filtered['PRODUCT1']
formatted_BAHRAIN['order_line/product_uom'] = "Unit" 
formatted_BAHRAIN['order_line/price_unit']  = BAHRAIN_df_filtered['TOTAL']
formatted_BAHRAIN['order_line/product_uom_qty remarks'] = BAHRAIN_df_filtered['REMARKS']


target_columns = [
    "client_order_ref", "customer_name", "partner_id", "whatsapp_no", "source_id", 
    "Pricelist Name", "street_no", "building_no", "zone_id (governarate)", 
    "wilayat_id", "city_id", "order_line/product_id", "order_line/product_uom", 
    "order_line/price_unit", "order_line/product_uom_qty remarks"
]

formatted_BAHRAIN = formatted_BAHRAIN[target_columns]


print(f"Total rows after cleaning: {len(formatted_BAHRAIN)}")
formatted_BAHRAIN.head(2)

Total rows after cleaning: 4


,client_order_ref,customer_name,partner_id,whatsapp_no,source_id,Pricelist Name,street_no,building_no,zone_id (governarate),wilayat_id,city_id,order_line/product_id,order_line/product_uom,order_line/price_unit,order_line/product_uom_qty remarks
804,EMC24609,Roy Samuel,37204254,37204254,SCENT PASSION - BAHRAIN,Default BHD pricelist,"5HM4+7F4, Tubli, Bahrain",,,,Tubli,OUD LOVERS,Unit,14,
835,EMC25577,Raju Joseph,39543122,39543122,SCENT PASSION - BAHRAIN,Default BHD pricelist,"6HF8+R34 Manama, Bahrain,26° 13' 28.2799"" N 50...",,,,Manama,ESENCIA FLORAL,Unit,15,


In [ ]:
safe_upload(target_spread, formatted_UAE, 'UAE_TAWSEEL')
safe_upload(target_spread, formatted_QATAR, 'FETCH_QATAR')
safe_upload(target_spread, formatted_BAHRAIN, 'FETCH_BAHRAIN')